# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark RDD - SOLUTION
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful

In [1]:
from pyspark import SparkContext, SparkConf
import numpy as np
import operator

In [2]:
conf=SparkConf().setAppName("Lab4-rdd").setMaster("local[*]")
sc = SparkContext(conf=conf)

Using PySpark and RDD's on the https://coding.csel.io machines is slow -- most of the code is executed in Python and this is much less efficient than the java-based code using the PySpark dataframes. Be patient and trying using `.cache()` to cache the output of joins. You may want to start with a reduced set of data before running the full task. You can use the `sample()` method to extract just a sample of the data or use 

These two RDD's are called "rawCitations" and "rawPatents" because you probably want to process them futher (e.g. convert them to integer types, etc). 

The `textFile` function returns data in strings. This should work fine for this lab.

Other methods you use might return data in type `Byte`. If you haven't used Python `Byte` types before, google it. You can convert a value of `x` type byte into e.g. a UTF8 string using `x.decode('uft-8')`. Alternatively, you can use the `open` method of the gzip library to read in all the lines as UTF-8 strings like this:
```
import gzip
with gzip.open('cite75_99.txt.gz', 'rt',encoding='utf-8') as f:
    rddCitations = sc.parallelize( f.readlines() )
```
This is less efficient than using `textFile` because `textFile` would use the underlying HDFS or other file system to read the file across all the worker nodes while the using `gzip.open()...readlines()` will read all the data in the frontend and then distribute it to all the worker nodes.

In [3]:
rddCitations = sc.textFile("cite75_99.txt.gz")
rddPatents = sc.textFile("apat63_99.txt.gz")

The data looks like the following.

In [4]:
rddCitations.take(5)

['"CITING","CITED"',
 '3858241,956203',
 '3858241,1324234',
 '3858241,3398406',
 '3858241,3557384']

In [5]:
rddPatents.take(5)

['"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD"',
 '3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,',
 '3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,',
 '3070803,1963,1096,,"US","IL",,1,,2,6,63,,9,,0.3704,,,,,,,',
 '3070804,1963,1096,,"US","OH",,1,,2,6,63,,3,,0.6667,,,,,,,']

In other words, they are a single string with multiple CSV's. You will need to convert these to (K,V) pairs, probably convert the keys to `int` and so on. You'll need to `filter` out the header string as well since there's no easy way to extract all the lines except the first.

In [6]:
citation_header = rddCitations.first()

citations_rdd = (
    rddCitations
    .filter(lambda line: line != citation_header)
    .map(lambda line: line.split(","))
    .map(lambda row: (int(row[0]), int(row[1])))
)

citations_rdd.take(5)

[(3858241, 956203),
 (3858241, 1324234),
 (3858241, 3398406),
 (3858241, 3557384),
 (3858241, 3634889)]

In [7]:
patent_header = rddPatents.first()

patents_rdd = (
    rddPatents
    .filter(lambda line: line != patent_header)
    .map(lambda line: line.split(","))
)

patent_states_rdd = patents_rdd.map(
    lambda row: (
        int(row[0]),
        row[5].replace('"', '') or None
    )
)

patent_states_rdd.take(5)

[(3070801, None),
 (3070802, 'TX'),
 (3070803, 'IL'),
 (3070804, 'OH'),
 (3070805, 'CA')]

In [8]:
citations_by_cited = citations_rdd.map(
    lambda x: (x[1], x[0])
)

In [9]:
cited_join = citations_by_cited.join(patent_states_rdd)

cited_join.take(5)

[(3665514, (3859666, 'MA')),
 (3665514, (3897597, 'MA')),
 (3665514, (3991422, 'MA')),
 (3665514, (4023209, 'MA')),
 (3665514, (4058854, 'MA'))]

In [10]:
citations_by_citing = cited_join.map(
    lambda x: (
        x[1][0],
        (x[0], x[1][1])
    )
)

In [11]:
state_join = citations_by_citing.join(patent_states_rdd)

state_join.take(5)

[(5770827, ((5464956, 'IL'), None)),
 (5770827, ((5015810, 'MO'), None)),
 (5770827, ((4259554, 'PA'), None)),
 (5770827, ((4410778, 'PA'), None)),
 (5770827, ((5003138, None), None))]

In [12]:
same_state_rdd = state_join.filter(
    lambda x:
        x[1][0][1] is not None and
        x[1][1] is not None and
        x[1][0][1] == x[1][1]
)

same_state_rdd.take(10)

[(5279088, ((3228158, 'CA'), 'CA')),
 (5279088, ((3784312, 'CA'), 'CA')),
 (5279088, ((4614013, 'CA'), 'CA')),
 (5279088, ((4894969, 'CA'), 'CA')),
 (4444905, ((3232887, 'IL'), 'IL')),
 (5198055, ((5017021, 'WI'), 'WI')),
 (5198055, ((4782951, 'WI'), 'WI')),
 (4663305, ((4219447, 'LA'), 'LA')),
 (4494087, ((4340870, 'AZ'), 'AZ')),
 (4494087, ((3688219, 'AZ'), 'AZ'))]

In [13]:
same_state_counts_rdd = (
    same_state_rdd
    .map(lambda x: (x[0], 1))
    .reduceByKey(lambda a, b: a + b)
)

same_state_counts_rdd.take(10)

[(4549218, 3),
 (4343616, 3),
 (4252554, 3),
 (4990260, 1),
 (4588929, 1),
 (5990454, 28),
 (5962815, 67),
 (5562024, 1),
 (5702601, 6),
 (5539437, 3)]

In [14]:
patents_by_number = patents_rdd.map(
    lambda row: (int(row[0]), row)
)

augmented_rdd = patents_by_number.leftOuterJoin(
    same_state_counts_rdd
)

augmented_rdd = augmented_rdd.map(
    lambda x: x[1][0] + [
        x[1][1] if x[1][1] is not None else 0
    ]
)

In [15]:
augmented_rdd.take(5)

[['3171904',
  '1965',
  '1887',
  '',
  '"FR"',
  '""',
  '',
  '1',
  '',
  '381',
  '4',
  '49',
  '',
  '5',
  '',
  '0.32',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  0],
 ['3173068',
  '1965',
  '1894',
  '',
  '"DE"',
  '""',
  '',
  '2',
  '',
  '257',
  '4',
  '46',
  '',
  '1',
  '',
  '0',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  0],
 ['3198484',
  '1965',
  '2041',
  '',
  '"US"',
  '"MA"',
  '',
  '2',
  '',
  '251',
  '5',
  '53',
  '',
  '3',
  '',
  '0.6667',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  0],
 ['3269244',
  '1966',
  '2433',
  '',
  '"US"',
  '"WI"',
  '',
  '2',
  '',
  '83',
  '5',
  '51',
  '',
  '0',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  0],
 ['3311980',
  '1967',
  '2650',
  '1963',
  '"US"',
  '"NC"',
  '',
  '1',
  '',
  '33',
  '6',
  '69',
  '',
  '0',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  0]]

In [16]:
top10_rdd = augmented_rdd.takeOrdered(
    10,
    key=lambda row: -row[-1]
)

for row in top10_rdd:
    print(row)

['5959466', '1999', '14515', '1997', '"US"', '"CA"', '5310', '2', '', '326', '4', '46', '159', '0', '1', '', '0.6186', '', '4.8868', '0.0455', '0.044', '', '', 125]
['5983822', '1999', '14564', '1998', '"US"', '"TX"', '569900', '2', '', '114', '5', '55', '200', '0', '0.995', '', '0.7201', '', '12.45', '0', '0', '', '', 103]
['6008204', '1999', '14606', '1998', '"US"', '"CA"', '749584', '2', '', '514', '3', '31', '121', '0', '1', '', '0.7415', '', '5', '0.0085', '0.0083', '', '', 100]
['5952345', '1999', '14501', '1997', '"US"', '"CA"', '749584', '2', '', '514', '3', '31', '118', '0', '1', '', '0.7442', '', '5.1102', '0', '0', '', '', 98]
['5958954', '1999', '14515', '1997', '"US"', '"CA"', '749584', '2', '', '514', '3', '31', '116', '0', '1', '', '0.7397', '', '5.181', '0', '0', '', '', 96]
['5998655', '1999', '14585', '1998', '"US"', '"CA"', '', '1', '', '560', '1', '14', '114', '0', '1', '', '0.7387', '', '5.1667', '', '', '', '', 96]
['5936426', '1999', '14466', '1997', '"US"', '"CA

Explanation

In RDD-based approach, the first step involves transformation of the citations and patents datasets into key-value pairs. The citations dataset is first joined with the dataset of patent-states to get the state of the cited patent and then once again joined to get the state of the citing patent. Only those citations where both states are present and identical are retained and counted for each citing patent. The resulting counts are left-joined to the original patent dataset such that all those patents that do not have any citations in the same state will have a count of 0.